# Phase 2 — Learning curves and efficiency benchmark

This notebook creates the final Task B (17-class) learning-curve and efficiency artifacts. It never downloads checkpoints and never retrains a model. Learning curves are parsed from literal Phase 1 notebook stdout; efficiency uses the five local seed-42 best checkpoints. Training/official metric evaluation (RTX 4090) and this local efficiency profile (RTX 3060) are recorded as separate environments.

In [1]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = Path.cwd()

if not (ROOT / 'src').exists():
    candidates = [ROOT, *ROOT.parents]
    ROOT = next((path for path in candidates if (path / 'src').is_dir() and (path / 'notebooks').is_dir()), ROOT)

sys.path.insert(0, str(ROOT))
print(f'ROOT={ROOT.resolve()}')

ROOT=/home/sakana/Code/ManifoldsTeam/woodbench


In [2]:
import importlib.util

MODULE_PATH = ROOT / 'notebooks' / 'phase 2' / 'phase2_benchmark.py'
spec = importlib.util.spec_from_file_location('phase2_benchmark', MODULE_PATH)
phase2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(phase2)
print(f'Loaded {MODULE_PATH.relative_to(ROOT)}')

Loaded notebooks/phase 2/phase2_benchmark.py


## 1. Learning curves

The five seed-42 main runs are recovered. Seed 42 is used consistently because the stored ResNet-50 seed-456 output contains repeated attempts after a disk-full failure and none matches the final checkpoint summary. A log is accepted only when its epoch count and final/best metrics match `test_metrics.json`; no values are interpolated.

In [3]:
methods_metadata = phase2.build_methods_artifacts(ROOT)
print('Generated and validated dataset/method artifacts from the split manifest.')
learning_curves, learning_summary, learning_metadata = phase2.build_learning_curves(ROOT)
print(f'Learning rows: {len(learning_curves):,}')
display(learning_curves.groupby(['display_name', 'seed'])['epoch'].max().unstack())
display(learning_curves.head())

Generated and validated dataset/method artifacts from the split manifest.


Learning rows: 194


seed,42
display_name,
ConvNeXt-Tiny,24
EfficientNet-B3,50
MobileNetV3-Large,50
ResNet-50,50
Swin-Tiny,20


,run_id,backbone,display_name,seed,experiment,epoch,epoch_budget,train_loss,train_acc,val_loss,val_acc,lr,source_notebook,source_metrics
150,taskB17__main__seed_42__convnext_tiny,convnext_tiny,ConvNeXt-Tiny,42,main,1,50,3.2412,2.86,3.2064,5.20,0.00002,notebooks/phase 1/task-B-17-new/04_train_convn...,data/experiments/Task-B-new/19-08-2026/main/se...
151,taskB17__main__seed_42__convnext_tiny,convnext_tiny,ConvNeXt-Tiny,42,main,2,50,1.1069,83.01,0.7182,96.46,0.00004,notebooks/phase 1/task-B-17-new/04_train_convn...,data/experiments/Task-B-new/19-08-2026/main/se...
152,taskB17__main__seed_42__convnext_tiny,convnext_tiny,ConvNeXt-Tiny,42,main,3,50,0.6656,98.27,0.7226,94.13,0.00006,notebooks/phase 1/task-B-17-new/04_train_convn...,data/experiments/Task-B-new/19-08-2026/main/se...
153,taskB17__main__seed_42__convnext_tiny,convnext_tiny,ConvNeXt-Tiny,42,main,4,50,0.6464,98.27,0.6386,98.01,0.00008,notebooks/phase 1/task-B-17-new/04_train_convn...,data/experiments/Task-B-new/19-08-2026/main/se...
154,taskB17__main__seed_42__convnext_tiny,convnext_tiny,ConvNeXt-Tiny,42,main,5,50,0.6450,98.07,0.6539,97.45,0.00010,notebooks/phase 1/task-B-17-new/04_train_convn...,data/experiments/Task-B-new/19-08-2026/main/se...


## 2. Efficiency on the local RTX 3060

Protocol: seed 42, batch size 1, 224×224, FP32, 50 warm-up passes and 200 timed passes. Accuracy/F1 are joined from the official RTX 4090 evaluation registry; only complexity, latency, model size, and VRAM are measured here. This cell fails instead of silently falling back to CPU or another GPU.

In [4]:
environment = phase2.live_environment()
display(environment)
efficiency, efficiency_metadata = phase2.run_efficiency_profile(ROOT, warmup=50, repetitions=200)
display(efficiency)

{'hostname': 'archlinux',
 'platform': 'Linux-7.2.2-arch1-1-x86_64-with-glibc2.44',
 'cpu': 'Architecture:                            x86_64\nCPU op-mode(s):                          32-bit, 64-bit\nAddress sizes:                           39 bits physical, 48 bits virtual\nByte Order:                              Little Endian\nCPU(s):                                  20\nOn-line CPU(s) list:                     0-19\nVendor ID:                               GenuineIntel\nModel name:                              Intel(R) Core(TM) i5-14600KF\nCPU family:                              6\nModel:                                   183\nThread(s) per core:                      2\nCore(s) per socket:                      14\nSocket(s):                               1\nStepping:                                1\nMicrocode version:                       0x137\nCPU(s) scaling MHz:                      54%\nCPU max MHz:                             5300.0000\nCPU min MHz:                          

Profiling ResNet-50 on NVIDIA GeForce RTX 3060 ...


Profiling EfficientNet-B3 on NVIDIA GeForce RTX 3060 ...


Profiling MobileNetV3-Large on NVIDIA GeForce RTX 3060 ...


Profiling ConvNeXt-Tiny on NVIDIA GeForce RTX 3060 ...


Profiling Swin-Tiny on NVIDIA GeForce RTX 3060 ...


,profile_id,source_run_id,backbone,display_name,seed,num_classes,image_size,batch_size,precision,total_params,...,latency_min_ms,latency_max_ms,warmup_runs,timed_runs,baseline_allocated_vram_mib,peak_allocated_vram_mib,incremental_peak_vram_mib,test_accuracy,test_f1_macro,checkpoint_path
0,efficiency__seed_42__resnet50,taskB17__main__seed_42__resnet50,resnet50,ResNet-50,42,17,224,1,FP32,23542865,...,2.835456,4.637024,50,200,98.744141,109.496582,10.752441,0.937970,0.939438,data/experiments/Task-B-new/19-08-2026/main/se...
1,efficiency__seed_42__efficientnet_b3,taskB17__main__seed_42__efficientnet_b3,efficientnet_b3,EfficientNet-B3,42,17,224,1,FP32,10722361,...,2.637824,4.819776,50,200,50.859863,71.262207,20.402344,0.973684,0.974049,data/experiments/Task-B-new/19-08-2026/main/se...
2,efficiency__seed_42__mobilenetv3_large,taskB17__main__seed_42__mobilenetv3_large,mobilenetv3_large,MobileNetV3-Large,42,17,224,1,FP32,4223809,...,0.857024,2.588448,50,200,24.982422,33.701172,8.718750,0.986842,0.987872,data/experiments/Task-B-new/19-08-2026/main/se...
3,efficiency__seed_42__convnext_tiny,taskB17__main__seed_42__convnext_tiny,convnext_tiny,ConvNeXt-Tiny,42,17,224,1,FP32,27833201,...,3.117856,4.388672,50,200,131.080078,146.267578,15.187500,0.998120,0.998267,data/experiments/Task-B-new/19-08-2026/main/se...
4,efficiency__seed_42__swin_tiny,taskB17__main__seed_42__swin_tiny,swin_tiny,Swin-Tiny,42,17,224,1,FP32,27532427,...,3.956832,5.236896,50,200,133.186523,150.624023,17.437500,0.996241,0.996560,data/experiments/Task-B-new/19-08-2026/main/se...


## 3. Validate handoff

Artifacts are written under `../WoodID/artifact/person2/learning_curves/` and `../WoodID/artifact/person2/efficiency/`.

In [5]:
validation = phase2.validate_outputs(ROOT)
print(f"Validated {len(validation['files'])} non-empty artifacts at {validation['validated_utc']}")
print(validation['integrated_tables']['status'])
for path in validation['files']:
    print(path)

Validated 16 non-empty artifacts at 2026-09-05T04:19:00Z
all integrated numeric cells match the official 19-Aug registry
/home/sakana/Code/ManifoldsTeam/WoodID/artifact/person2/methods/tab_species.tex
/home/sakana/Code/ManifoldsTeam/WoodID/artifact/person2/methods/tab_species_full.tex
/home/sakana/Code/ManifoldsTeam/WoodID/artifact/person2/methods/tab_augconfig.tex
/home/sakana/Code/ManifoldsTeam/WoodID/artifact/person2/methods/methods_sources.json
/home/sakana/Code/ManifoldsTeam/WoodID/artifact/person2/learning_curves/learning_curves.csv
/home/sakana/Code/ManifoldsTeam/WoodID/artifact/person2/learning_curves/learning_curves_summary.csv
/home/sakana/Code/ManifoldsTeam/WoodID/artifact/person2/learning_curves/learning_curves_metadata.json
/home/sakana/Code/ManifoldsTeam/WoodID/artifact/person2/learning_curves/fig_learning_curves.pdf
/home/sakana/Code/ManifoldsTeam/WoodID/artifact/person2/learning_curves/fig_learning_curves.png
/home/sakana/Code/ManifoldsTeam/WoodID/artifact/person2/effic